In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from datetime import datetime
from sklearn import metrics
from sklearn.impute import KNNImputer
from sklearn import preprocessing
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import GridSearchCV
from mord import LogisticIT,LogisticAT
from tqdm import tqdm
from sklearn.model_selection import KFold, StratifiedKFold

In [ ]:
def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    """
    given a sklearn confusion matrix (cm), make a nice plot

    Arguments
    ---------
    cm:           confusion matrix from sklearn.metrics.confusion_matrix

    target_names: given classification classes such as [0, 1, 2]
                  the class names, for example: ['high', 'medium', 'low']

    title:        the text to display at the top of the matrix

    cmap:         the gradient of the values displayed from matplotlib.pyplot.cm
                  see http://matplotlib.org/examples/color/colormaps_reference.html
                  plt.get_cmap('jet') or plt.cm.Blues

    normalize:    If False, plot the raw numbers
                  If True, plot the proportions

    Usage
    -----
    plot_confusion_matrix(cm           = cm,                  # confusion matrix created by
                                                              # sklearn.metrics.confusion_matrix
                          normalize    = True,                # show proportions
                          target_names = y_labels_vals,       # list of names of the classes
                          title        = best_estimator_name) # title of graph

    Citiation
    ---------
    http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html

    """
    import matplotlib.pyplot as plt
    import numpy as np
    import itertools

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

In [ ]:
def balance_data(X,Y,class_size):
    n_classes = len(Y.unique())
    class_list = Y.unique()
    df = pd.concat([X,Y],axis=1)
    balanced_df = pd.DataFrame(columns = df.columns)
    for n in range(n_classes): 
        c = class_list[n]
        class_df = df[df['class'] == c]
        if len(class_df) > class_size:
            # Downsample majority class
            balanced_class_df = resample(class_df,replace=False,n_samples=class_size) 
        elif len(class_df) < class_size:
            #oversampe minority class
            balanced_class_df = resample(class_df,replace=True,n_samples=class_size) 
        balanced_df = balanced_df.append(balanced_class_df)    
    balanced_df = balanced_df.sample(frac=1).reset_index(drop=True)
    balanced_X = balanced_df[balanced_df.columns[:-1]]
    balanced_Y = balanced_df[balanced_df.columns[-1]]
    balanced_Y=balanced_Y.astype('int')
    return balanced_X, balanced_Y

# Load Data

In [ ]:
study_criteria_table = pd.read_excel(r"../medical_data/study_criteria_table_label_V6.xlsx")

In [ ]:
criteria_df = study_criteria_table 

In [ ]:
study_features_table = pd.read_csv(r"../eeg_data/study_features_table_v4.csv")

In [ ]:
CDR_EDW_df = pd.read_excel(r'../medical_data/CDR_EDW_Final.xlsx')
CDR_RPDR_df = pd.read_excel(r'../medical_data/CDR_RPDR_Final.xlsx')

In [ ]:
#for index,row in CDR_RPDR_df.iterrows():
#    if type(row['LMRNote_Date']) is datetime:
#        CDR_RPDR_df.at[index,'LMRNote_Date'] = row['LMRNote_Date'].strftime('%Y-%m-%d')

# Preprocess Data

In [ ]:
data_df = study_criteria_table[['FolderName','Predicted_Stage']].merge(study_features_table,on=['FolderName'])

In [ ]:
data_df = data_df.drop_duplicates(subset=['FolderName'])

In [ ]:
data_df

In [ ]:
CDR_RPDR_df = CDR_RPDR_df.sort_values(by='CDRScore',ascending=False).dropna(subset=['CDRScore']).drop_duplicates('EMPI')
CDR_EDW_df = CDR_EDW_df.sort_values(by='CDRScore',ascending=False).dropna(subset=['CDRScore']).drop_duplicates('PatientID')

In [ ]:
CDR_RPDR_df = CDR_RPDR_df[CDR_RPDR_df['Correct'] == 'Y']
CDR_EDW_df = CDR_EDW_df[CDR_EDW_df['Correct'] == 'Y']

In [ ]:
criteria_df.columns

In [ ]:
CDR_df = pd.DataFrame(columns=['FolderName','CDR','dT'])
CDR_df[['FolderName','PatientID','CDR','dT','Stage']] = criteria_df[['FolderName','PatientID','CDR_Score','CDR_dT','Predicted_Stage']]

In [ ]:
for index,row in tqdm(criteria_df.iterrows(),total=criteria_df.shape[0],position=0,leave=False):
    DoV = row['DateOfVisit']
    if len(CDR_EDW_df[CDR_EDW_df['PatientID'] == row['PatientID']])>0:
        df = CDR_EDW_df[CDR_EDW_df['PatientID'] == row['PatientID']]
        df['CDR_dT'] = [(datetime.strptime( DoV ,'%Y-%m-%d')-datetime.strptime(row['ContactDTSForNote'],'%Y-%m-%d')).days for index,row in df.iterrows()] 
        df = df[df['CDR_dT'] >= -360]
        if len(df) >0:
            df['abs_dT'] = abs(df['CDR_dT'])
            df = df.sort_values(by=['abs_dT'])
            CDR = df.iloc[0]['CDRScore']
            criteria_df.at[index,'CDR'] = CDR
            CDR_df.at[index,'dT'] = df.iloc[0]['CDR_dT']
    if len(CDR_RPDR_df[CDR_RPDR_df['EMPI'] == row['EMPI']])>0:
        df = CDR_RPDR_df[CDR_RPDR_df['EMPI'] == row['EMPI']]
        df['CDR_dT'] = [(datetime.strptime( DoV ,'%Y-%m-%d')-datetime.strptime(row['LMRNote_Date'],'%Y-%m-%d')).days for index,row in df.iterrows()] 
        df = df[df['CDR_dT'] >= -360]
        if len(df) > 0: 
            df['abs_dT'] = abs(df['CDR_dT'])
            df = df.sort_values(by=['abs_dT'])
            CDR = df.iloc[0]['CDRScore']
            CDR_df.at[index,'CDR'] = CDR
            CDR_df.at[index,'dT'] = df.iloc[0]['CDR_dT']

In [ ]:
CDR_df.to_csv('CDR_df.csv',index=False)

In [ ]:
CDR_df = pd.read_csv('CDR_df.csv')
CDR_df

In [ ]:
CDR_df.at[CDR_df['FolderName'].isin(study_criteria_table[study_criteria_table['Predicted_Stage'] == 'No Dementia']['FolderName']),'CDR'] = 0

In [ ]:
for score in CDR_df['CDR'].unique():
    print('CDR =',score,":",len(CDR_df[(CDR_df['CDR'] == score)&(CDR_df['Stage'] != 'Excluded')]))

In [ ]:
CDR_df = CDR_df.dropna(subset=['CDR'])
CDR_df  = CDR_df[(CDR_df['Stage'] != 'Excluded')]

In [ ]:
CDR_df = CDR_df.reset_index(drop=True)

In [ ]:
CDR_df = CDR_df[['FolderName', 'PatientID','CDR', 'dT', 'Stage']]

In [ ]:
CDR_df

In [ ]:
y = pd.concat([CDR_df[CDR_df['CDR'] == 0][['FolderName','CDR']].sample(200),CDR_df[CDR_df['CDR'] != 0][['FolderName','CDR']]])
y = y.sample(frac=1)
y=y.reset_index(drop=True)
X = y.merge(data_df,on=['FolderName'],how='left')
X = X[X.columns[3:]]
y = y[y.columns[1]]
y[y>=1] = 2
y[y==0.5] = 1
y[y==0] = 0
y = y.astype(int)

In [ ]:
[10**x for x in range(-3,5)]

# Ordinal Logistic Regression

In [ ]:
# configure the cross-validation procedure
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# enumerate splits
cm_total = [[0,0,0],[0,0,0],[0,0,0]]
auc_all = list()
kappa_all = list()
f1_all = list()
f= 0
y_test_all =np.array([])
y_prob_all =np.zeros([0,3])
y_pred_all =np.array([])
for train_idx, test_idx in cv_outer.split(X,y):
    f+=1
    # split data
    print('Computing Fold ' + str(f))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    #imputer after scaler
    scaler = preprocessing.StandardScaler().fit(X_train)
    X_train = pd.DataFrame(data=scaler.transform(X_train),columns=X_train.columns)
    imputer = KNNImputer(n_neighbors=10).fit(X_train)
    X_train = pd.DataFrame(data=imputer.transform(X_train),columns=X_train.columns)
    
    #scaler = preprocessing.StandardScaler().fit(X_test)
    #imputer = KNNImputer(n_neighbors=10).fit(X_test)
    X_test = pd.DataFrame(data=scaler.transform(X_test),columns=X_train.columns)
    X_test = pd.DataFrame(data=imputer.transform(X_test),columns=X_train.columns)
    
    # configure the cross-validation procedure
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    
    # define the model
    model = LogisticIT()
     
    params = {'max_iter': [100], 'alpha': [10**x for x in range(-3,5)]}
    gd_search = GridSearchCV(model, params, scoring='f1_weighted', n_jobs=-1, cv=cv_inner).fit(X_train, y_train)
    best_params = gd_search.best_params_
    best_model = gd_search.best_estimator_

    y_prob = best_model.predict_proba(X_test)
    y_pred =  best_model.predict(X_test)
    y_test_binarized =  label_binarize(y_test, classes=[0, 1, 2])
    
    auc = metrics.roc_auc_score(y_test_binarized, y_prob,multi_class = 'ovr')
    f1 = metrics.f1_score(y_test, y_pred,average='macro')
    
    # store the result
    auc_all.append(auc)
    f1_all.append(f1)
    y_test_all = np.concatenate((y_test_all, y_test))
    y_pred_all = np.concatenate((y_pred_all, y_pred))
    y_prob_all = np.concatenate((y_prob_all, y_prob))
    
    print("Val Auc:",auc, "Best GS Auc:",gd_search.best_score_, "Best Params:",gd_search.best_params_)
    print('Accuracy Score : ' + str(metrics.accuracy_score(y_test, y_pred)))
    print('Precision Score : ' + str(metrics.precision_score(y_test, y_pred,average='macro')))
    print('Recall Score : ' + str(metrics.recall_score(y_test, y_pred,average='macro')))
    print('F1 Score : ' + str(f1))
    kappa = metrics.cohen_kappa_score(y_test,y_pred)
    print('Kappa : ' + str(kappa))
    kappa_all.append(kappa)
    
    cm = metrics.confusion_matrix(y_test,y_pred,labels=[0, 1, 2])
    cm_total = cm + cm_total
    
# summarize the estimated performance of the model
print('Estimated AUC: %.3f (%.3f)' % (np.mean(auc_all), np.std(auc_all)))
print('Estimated Kappa: %.3f (%.3f)' % (np.mean(kappa_all), np.std(kappa_all)))
print('Estimated F1: %.3f (%.3f)' % (np.mean(f1_all), np.std(f1_all)))

In [ ]:
plot_confusion_matrix(cm=cm_total,
                          target_names=['0','0.5','>=1'],
                          title='Confusion matrix',
                          cmap=None,
                          normalize=False)
report= metrics.classification_report(y_test_all,y_pred_all)
print(report)

In [ ]:
plt.figure()
labels = ['>=1','0.5','0']
y_test_all_binarized = label_binarize(y_test_all, classes=[0,1,2])
for i in range(3):
    fpr, tpr, thresholds = metrics.roc_curve(y_test_all_binarized[:,i], y_prob_all[:,i]) 
    roc_auc = metrics.auc(fpr, tpr)
    plt.plot(fpr, tpr, label='ROC curve of Class ' + labels[i]  + ' (area = %0.2f)' % roc_auc)

plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Multiclass Classification of Dementia using Ordinal Logitics Regression (Immediate-Threshold Variant)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
#Final Model
scaler = preprocessing.StandardScaler().fit(X)
X_processed = pd.DataFrame(data=scaler.transform(X),columns=X.columns)
imputer = KNNImputer(n_neighbors=10).fit(X_processed)
X_processed = pd.DataFrame(data=imputer.transform(X_processed),columns=X.columns)
    
final_model = LogisticIT(alpha=1,max_iter=100)
final_model.fit(X_processed,y)


In [ ]:
# configure the cross-validation procedure
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# enumerate splits
cm_total = [[0,0,0],[0,0,0],[0,0,0]]
auc_all = list()
kappa_all = list()
f1_all = list()
f= 0
y_test_all =np.array([])
y_prob_all =np.zeros([0,3])
y_pred_all =np.array([])
for train_idx, test_idx in cv_outer.split(X,y):
    f+=1
    # split data
    print('Computing Fold ' + str(f))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    #imputer after scaler
    scaler = preprocessing.StandardScaler().fit(X_train)
    X_train = pd.DataFrame(data=scaler.transform(X_train),columns=X_train.columns)
    imputer = KNNImputer(n_neighbors=10).fit(X_train)
    X_train = pd.DataFrame(data=imputer.transform(X_train),columns=X_train.columns)
    
    #scaler = preprocessing.StandardScaler().fit(X_test)
    #imputer = KNNImputer(n_neighbors=10).fit(X_test)
    X_test = pd.DataFrame(data=scaler.transform(X_test),columns=X_train.columns)
    X_test = pd.DataFrame(data=imputer.transform(X_test),columns=X_train.columns)
    
    # configure the cross-validation procedure
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    
    # define the model
    model = LogisticAT()
     
    params = {'max_iter': [100], 'alpha': [10**x for x in range(-3,5)]}
    gd_search = GridSearchCV(model, params, scoring='f1_weighted', n_jobs=-1, cv=cv_inner).fit(X_train, y_train)
    best_params = gd_search.best_params_
    best_model = gd_search.best_estimator_

    y_prob = best_model.predict_proba(X_test)
    y_pred =  best_model.predict(X_test)
    y_test_binarized =  label_binarize(y_test, classes=[0, 1, 2,3])
    
    auc = metrics.roc_auc_score(y_test_binarized, y_prob,multi_class = 'ovr')
    f1 = metrics.f1_score(y_test, y_pred,average='macro')
    
    # store the result
    auc_all.append(auc)
    f1_all.append(f1)
    y_test_all = np.concatenate((y_test_all, y_test))
    y_pred_all = np.concatenate((y_pred_all, y_pred))
    y_prob_all = np.concatenate((y_prob_all, y_prob))
    
    print("Val Auc:",auc, "Best GS Auc:",gd_search.best_score_, "Best Params:",gd_search.best_params_)
    print('Accuracy Score : ' + str(metrics.accuracy_score(y_test, y_pred)))
    print('Precision Score : ' + str(metrics.precision_score(y_test, y_pred,average='macro')))
    print('Recall Score : ' + str(metrics.recall_score(y_test, y_pred,average='macro')))
    print('F1 Score : ' + str(f1))
    kappa = metrics.cohen_kappa_score(y_test,y_pred)
    print('Kappa : ' + str(kappa))
    kappa_all.append(kappa)
    
    cm = metrics.confusion_matrix(y_test,y_pred,labels=[0, 1, 2])
    cm_total = cm + cm_total
    
# summarize the estimated performance of the model
print('Estimated AUC: %.3f (%.3f)' % (np.mean(auc_all), np.std(auc_all)))
print('Estimated Kappa: %.3f (%.3f)' % (np.mean(kappa_all), np.std(kappa_all)))
print('Estimated F1: %.3f (%.3f)' % (np.mean(f1_all), np.std(f1_all)))

In [ ]:
plot_confusion_matrix(cm=cm_total,
                          target_names=['0','0.5','>=1'],
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True)
report= metrics.classification_report(y_test_all,y_pred_all)
print(report)

In [ ]:
plt.figure()
labels = ['>=1','0.5','0']
y_test_all_binarized = label_binarize(y_test_all, classes=[0,1,2])
for i in range(3):
    fpr, tpr, thresholds = metrics.roc_curve(y_test_all_binarized[:,i], y_prob_all[:,i]) 
    roc_auc = metrics.auc(fpr, tpr)
    plt.plot(fpr, tpr, label='ROC curve of Class ' + labels[i]  + ' (area = %0.2f)' % roc_auc)

plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Multiclass Classification of Dementia using Ordinal Logitics Regression (All-Threshold Variant)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
plt.figure()
labels = ['>=1','0.5','0']
y_test_all_binarized = label_binarize(y_test_all, classes=[0,1,2])
for i in range(3):
    fpr, tpr, thresholds = metrics.roc_curve(y_test_all_binarized[:,i], y_prob_all[:,i]) 
    roc_auc = metrics.auc(fpr, tpr)
    plt.plot(fpr, tpr, label='ROC curve of Class ' + labels[i]  + ' (area = %0.2f)' % roc_auc)

plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Multiclass Classification of Dementia using Ordinal Logitics Regression (All-Threshold Variant)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
y_prob_all

In [ ]:
y_test_all

In [ ]:
y_test_all_binarized